# Mono-$Z$ analizi — adım adım, gözleyerek

Bu defter `monoz_dark_matter_analizi.ipynb`'nin **rehberli** sürümüdür. Ana defter
her şeyi tek hücrede yapıp sonucu verir; burada ise her adım ayrı bir hücrededir ve
kararları **sen** verirsin.

Hücrelerde iki etiket var:

- **GÖZLE** — sadece çalıştır ve çıktıya bak, değiştirilecek bir şey yok.
- **KARAR** — hücrenin başında büyük harfle yazılmış bir değişken vardır
  (ör. `MET_MIN`, `BENIM_OZNITELIKLERIM`). Önceki çıktılara bakarak onu değiştir,
  sonra hücreyi ve sonrakileri yeniden çalıştır.

**Akış**

| Adım | Ne yapılıyor | Tür |
| --- | --- | --- |
| 1 | Örnekleri kesimsiz yükle | GÖZLE |
| 2 | Hangi değişkenler var? | GÖZLE |
| 3 | Değişken dağılımları: sinyal vs arka plan | GÖZLE / KARAR |
| 4 | Sinyal bölgesi kesimleri ve kesim akışı | KARAR |
| 5 | Olay ağırlıkları, negatif ağırlıklar | GÖZLE |
| 6 | Eğitim / doğrulama / test bölünmesi | GÖZLE |
| 7 | Aday havuzunun tek değişkenli ayırt ediciliği | GÖZLE |
| 8 | Korelasyon ve fazlalık | GÖZLE |
| 9 | **Öznitelik seçimi** | KARAR |
| 10 | Seçimini doğrulama kümesinde dene, önem sıralaması | GÖZLE (→ 9'a dön) |
| 11 | Son rapor: test kümesi, yalnızca bir kez | GÖZLE |

**Önemli kural (sızıntı).** Adım 6'dan itibaren bütün keşif **yalnızca eğitim
kümesinde**, karşılaştırma **doğrulama kümesinde** yapılır. Test kümesine yalnızca
Adım 11'de, seçimin bittikten sonra bir kez bakılır. Test sonucuna bakıp geri dönüp
öznitelik değiştirirsen test kümesi artık "görülmemiş" veri değildir.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

LUMI = 2320.0          # pb^-1, CMS Run 2015D
EXTRACT = '/work/_extract.py'   # ana defterle ayni yukleyici
exec(open(EXTRACT).read())      # load, BKG, SIG, cell_resample

pd.set_option('display.width', 160)
pd.set_option('display.max_rows', 80)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
PROC = ['DY', 'ZZ', 'WZ', 'ttbar', 'tW_top', 'tW_antitop']
PROC_RENK = dict(zip(PROC, ['#4878a8', '#d1495b', '#48a860', '#e8a33d',
                            '#9b6bbf', '#8c6d5c']))
print('arka planlar:', list(BKG))
print('sinyal noktalari:', list(SIG))

## Adım 1 — Örnekleri kesimsiz yükle

**KARAR:** hangi sinyal noktasıyla çalışacaksın? Ana defter
`'axial (50, 200)'` kullanır. Yukarıdaki listeden başka birini de seçebilirsin.

Burada henüz **hiçbir sinyal bölgesi kesimi yoktur**; yükleyicinin ön seçiminden
(iki lepton, $Z$ adayı) geçen olayların tamamı gelir.

In [ ]:
SINYAL = 'axial (50, 200)'

ham = pd.concat([load({k: v}).assign(src=k) for k, v in BKG.items()],
                ignore_index=True)
sig_ham = load({'A': SIG[SINYAL]}).assign(src='sinyal')

ozet = (pd.concat([ham, sig_ham])
        .groupby('src', sort=False)
        .agg(olay=('weight', 'size'), verim=('weight', 'sum')))
ozet['verim'] *= LUMI
ozet['olay basina'] = ozet.verim / ozet.olay
print(ozet.round(3).to_string())

**GÖZLE:** `olay` ham MC olay sayısı, `verim` ise $\sigma\mathcal{L}$ ile normalize
edilmiş beklenen olay sayısı. `olay basina` sütununda DY'nin ne kadar ağır
olduğuna dikkat et (ana defter §1a: DY $\sim41$, $ZZ$ $\sim0.02$).

## Adım 2 — Hangi değişkenler var?

Yükleyici her olay için bir sürü sütun üretir. Burada hepsini, türü, aralığı ve
kaç farklı değer aldığıyla birlikte listeliyoruz. Sabit (tek değerli) ya da `NaN`
içeren değişkenler model girdisi olamaz.

In [ ]:
DISLA = {'weight', 'src'}          # ozellik olmayan sutunlar

def sutun_tablosu(df):
    satir = []
    for c in df.columns:
        if c in DISLA or not np.issubdtype(df[c].dtype, np.number):
            continue
        x = df[c]
        satir.append({'degisken': c, 'tur': str(x.dtype),
                      'min': x.min(), 'medyan': x.median(), 'max': x.max(),
                      'farkli_deger': x.nunique(), 'NaN': int(x.isna().sum())})
    return pd.DataFrame(satir).set_index('degisken')

tablo = sutun_tablosu(ham)
print(f'{len(tablo)} sayisal degisken')
print(tablo.round(3).to_string())
sabit = tablo.index[tablo.farkli_deger <= 1].tolist()
print('\nsabit (kullanilamaz):', sabit or 'yok')

**GÖZLE:**

- `farkli_deger` küçükse (ör. `n_jets`, `has_jet1`) değişken **ayrıktır**.
  BDT/DNN için sorun değil; NSF için dequantize gerekir (ana defter §3).
- `met_phi`, `Z_phi` gibi açılar tek başına fizik taşımaz (detektör simetrik);
  bunların *farkı* (`dphi_...`) anlamlıdır. Adım 7'de bunu AUC'de göreceksin.
- `has_jet1` gibi bayraklar `n_jets` ile neredeyse aynı bilgiyi taşır (Adım 8).

## Adım 3 — Değişken dağılımları: sinyal vs arka plan

`ciz('degisken')` bir değişkenin şeklini gösterir:

- **sol:** her süreç ayrı ayrı, **birim alana normalize** (şekil karşılaştırması;
  süreç ne kadar baskın olursa olsun aynı ölçekte görünür),
- **sağ:** fiziksel verimle yığılı arka plan + sinyal (log ölçek; gerçekte ne
  göreceğin).

**KARAR:** `GOSTER` listesine merak ettiğin değişkenleri yaz. İki eğrinin
**ayrıştığı** değişkenler aday öznitelik, **üst üste binenler** büyük ihtimalle
işe yaramaz.

In [ ]:
def ciz(f, df_b=None, df_s=None, bins=40, aralik=None):
    df_b = ham if df_b is None else df_b
    df_s = sig_ham if df_s is None else df_s
    hepsi = np.r_[df_b[f], df_s[f]]
    lo, hi = aralik or np.percentile(hepsi, [0.5, 99.5])
    if lo == hi:
        lo, hi = lo - .5, hi + .5
    kenar = np.linspace(lo, hi, bins + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    for p in PROC:
        d = df_b[df_b.src == p]
        if len(d) > 20:
            ax[0].hist(d[f], bins=kenar, density=True, histtype='step',
                       lw=1.4, color=PROC_RENK[p], label=p)
    ax[0].hist(df_s[f], bins=kenar, density=True, histtype='step', lw=2.4,
               color='crimson', label='sinyal')
    ax[0].set_title(f'{f}: sekil (birim alan)')
    ax[0].legend(fontsize=8, ncol=2, frameon=False)
    var = [p for p in PROC if (df_b.src == p).any()]
    ax[1].hist([df_b[df_b.src == p][f] for p in var], bins=kenar,
               weights=[df_b[df_b.src == p].weight * LUMI for p in var],
               stacked=True, color=[PROC_RENK[p] for p in var], label=var)
    ax[1].hist(df_s[f], bins=kenar, weights=df_s.weight * LUMI,
               histtype='step', lw=2.4, color='crimson', label='sinyal')
    ax[1].set_yscale('log')
    ax[1].set_title(f'{f}: verim (yigili)')
    for a in ax:
        a.set_xlabel(f)
        a.grid(alpha=.3)
    plt.tight_layout()
    plt.show()


GOSTER = ['met', 'mT', 'Z_pt', 'dphi_Z_met', 'n_jets', 'm_ll']
for f in GOSTER:
    ciz(f)

**GÖZLE — neye bakmalı?**

- `met`: sinyal neden sağa kayık? ($\chi\bar\chi$ görünmez, $Z$'ye karşı geri teper.)
- `dphi_Z_met`: sinyalde $\pi$'ye yığılma = sırt sırta topoloji.
- `n_jets`: $t\bar t$ ve $tW$ çok jetli; jet vetosu neden işe yarar?
- `m_ll`: herkes $Z$ tepesinde mi? (Evet; o yüzden ayırt edicilik düşük beklenir.)

Bu gözlemler Adım 4'teki kesimleri seçmeni sağlar.

## Adım 4 — Sinyal bölgesi (SR) kesimleri

Ana defter Rasineni & Chebrolu'nun kesimlerini kullanır:
$E_T^{\rm miss}\ge50$ GeV, $|\Delta\phi(Z,E_T^{\rm miss})|>2.5$, $n_{\rm jet}\le1$.

**KARAR:** aşağıdaki sayıları değiştirip kesim akışının nasıl değiştiğine bak.
`None` yazarsan o kesim uygulanmaz. Tablonun son sütunu kaba bir anlamlılık
($Z_A$, Asimov) verir.

*Not:* kesimleri burada bütün MC üzerinde ayarlıyoruz; bu kaba, ön seçim
düzeyinde bir karardır. Sonraki makine öğrenmesi adımları bölünmeyle korunur.

In [ ]:
MET_MIN = 50          # GeV
DPHI_MIN = 2.5        # rad
NJET_MAX = 1
MLL_PENCERE = None    # ornek: 15  ->  |m_ll - 91.19| < 15 GeV

def asimov(s, b):
    if s <= 0 or b <= 0:
        return 0.0
    return np.sqrt(2 * ((s + b) * np.log1p(s / b) - s))


def kesimler():
    k = []
    if MET_MIN is not None:
        k.append((f'met >= {MET_MIN}', lambda d: d.met >= MET_MIN))
    if DPHI_MIN is not None:
        k.append((f'dphi_Z_met > {DPHI_MIN}', lambda d: d.dphi_Z_met > DPHI_MIN))
    if NJET_MAX is not None:
        k.append((f'n_jets <= {NJET_MAX}', lambda d: d.n_jets <= NJET_MAX))
    if MLL_PENCERE is not None:
        k.append((f'|m_ll-mZ| < {MLL_PENCERE}',
                  lambda d: (d.m_ll - 91.1876).abs() < MLL_PENCERE))
    return k


def kesim_akisi(b, s):
    satir, mb, ms = [], np.ones(len(b), bool), np.ones(len(s), bool)
    for ad, f in [('on secim', lambda d: np.ones(len(d), bool))] + kesimler():
        mb &= np.asarray(f(b))
        ms &= np.asarray(f(s))
        r = {'kesim': ad}
        for p in PROC:
            r[p] = b.weight[mb & (b.src == p).to_numpy()].sum() * LUMI
        B, S = b.weight[mb].sum() * LUMI, s.weight[ms].sum() * LUMI
        r.update(B=B, S=S, ham_B=int(mb.sum()), ham_S=int(ms.sum()),
                 Z_A=asimov(S, B))
        satir.append(r)
    return pd.DataFrame(satir).set_index('kesim'), mb, ms


akis, m_b, m_s = kesim_akisi(ham, sig_ham)
print(akis.round(2).to_string())
bkg = ham[m_b].reset_index(drop=True)
sig = sig_ham[m_s].reset_index(drop=True)
print(f'\nSR: arka plan {len(bkg)} olay, sinyal {len(sig)} olay')

**GÖZLE:** Her satırda hangi süreç en çok azalıyor? Örneğin jet vetosu
$t\bar t$'yi, $\Delta\phi$ kesimi DY'yi bastırmalı. `ham_B` çok küçülürse
(birkaç yüz olay) makine öğrenmesi için istatistik kalmaz — kesimi gevşet.

Kesimden sonra dağılımlara tekrar bakmak istersen: `ciz('met', bkg, sig)`.

## Adım 5 — Olay ağırlıkları

DY NLO (`amcatnloFXFX`) olduğu için bazı olayların ağırlığı **negatiftir**. Ana
defter bunları hücre yeniden örneklemesi (`cell_resample`) ile eritir: komşu
olaylarla toplam pozitif olana kadar hücre kurulur, toplam ağırlık korunur.

$N_{\rm eff}=(\sum w)^2/\sum w^2$ : ağırlıklı bir örneğin "kaç eşit ağırlıklı
olaya denk" olduğunu söyler.

In [ ]:
RESAMPLE = ['met', 'mT', 'Z_pt', 'dphi_Z_met', 'dR_ll', 'n_jets']

def n_eff(w):
    q = (w ** 2).sum()
    return w.sum() ** 2 / q if q > 0 else 0.0


once = bkg.weight.to_numpy().copy()
bkg['weight'] = cell_resample(bkg, RESAMPLE)
sonra = bkg.weight.to_numpy()

tab = []
for p in PROC:
    m = (bkg.src == p).to_numpy()
    if m.any():
        tab.append({'surec': p, 'olay': int(m.sum()),
                    'negatif_once': int((once[m] < 0).sum()),
                    'negatif_sonra': int((sonra[m] < 0).sum()),
                    'Neff_once': n_eff(once[m]), 'Neff_sonra': n_eff(sonra[m]),
                    'verim_once': once[m].sum() * LUMI,
                    'verim_sonra': sonra[m].sum() * LUMI})
print(pd.DataFrame(tab).round(2).to_string(index=False))
print(f'\ntoplam verim: {once.sum() * LUMI:.2f} -> {sonra.sum() * LUMI:.2f}'
      '  (korunmali)')

**GÖZLE:** DY'nin `olay` sayısı ile `Neff`'i arasındaki farka bak. Ana defterde
"ağırlıksız AUC neden düşük, ağırlıklı AUC neden yüksek" sorusunun cevabı bu
tablodadır: olay sayısında $ZZ$, verimde DY baskındır.

## Adım 6 — Bölünme (bundan sonrası için kural)

Sinyal ve arka plan **ayrı ayrı** %60 / %20 / %20 bölünür (ana defterle aynı
`split3`). Bundan sonra:

| küme | bu defterde kullanımı |
| --- | --- |
| `b_tr`, `s_tr` (eğitim) | Adım 7–8 keşfi, model eğitimi |
| `b_va`, `s_va` (doğrulama) | Adım 10: seçimleri karşılaştırma, eşik seçimi |
| `b_te`, `s_te` (test) | yalnızca Adım 11 |

In [ ]:
TOHUM = 0
FRAC = 0.2

def split3(df, seed):
    tr, rest = train_test_split(df, test_size=2 * FRAC, random_state=seed)
    va, te = train_test_split(rest, test_size=0.5, random_state=seed)
    return tr, va, te


b_tr, b_va, b_te = split3(bkg, TOHUM)
s_tr, s_va, s_te = split3(sig, TOHUM)
print(pd.DataFrame({'arka plan': [len(b_tr), len(b_va), len(b_te)],
                    'sinyal': [len(s_tr), len(s_va), len(s_te)]},
                   index=['egitim', 'dogrulama', 'test']).to_string())

## Adım 7 — Aday havuzu: her değişken tek başına ne kadar ayırıyor?

Sabit olmayan bütün sayısal sütunlar aday. Her biri için **eğitim kümesinde**:

- `AUC_verim`: fiziksel ağırlıklı tek değişken AUC'si (gerçek arka plan karışımı),
- `AUC_ham`: ağırlıksız (fiilen çoğunlukla $ZZ$'ye karşı),
- `yon`: sinyal yüksek değerlerde mi (`+`) düşük değerlerde mi (`-`).

AUC $0.5$ = hiç ayırmıyor, $1.0$ = mükemmel. Ana defterdeki 8'li ve 32'lik
havuz da işaretlendi.

In [ ]:
LITERATUR_8 = ['met', 'mT', 'Z_pt', 'dphi_Z_met',
               'dR_ll', 'm_ll', 'n_jets', 'met_over_ptZ']

ADAY = [c for c in sutun_tablosu(b_tr).index if b_tr[c].std() > 1e-9]
y_tr = np.r_[np.zeros(len(b_tr)), np.ones(len(s_tr))]
w_tr_fiz = np.abs(np.r_[b_tr.weight, s_tr.weight])

tek = []
for f in ADAY:
    x = np.r_[b_tr[f], s_tr[f]]
    a_w = roc_auc_score(y_tr, x, sample_weight=w_tr_fiz)
    a_h = roc_auc_score(y_tr, x)
    tek.append({'degisken': f, 'AUC_verim': max(a_w, 1 - a_w),
                'AUC_ham': max(a_h, 1 - a_h), 'yon': '+' if a_w >= .5 else '-',
                'literatur8': f in LITERATUR_8})
tek = pd.DataFrame(tek).set_index('degisken').sort_values('AUC_verim',
                                                          ascending=False)
print(tek.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 0.26 * len(tek) + 1))
renk = ['#d1495b' if v else '#4878a8' for v in tek.literatur8]
ax.barh(tek.index[::-1], tek.AUC_verim[::-1], color=renk[::-1])
ax.axvline(.5, color='gray', ls=':')
ax.set_xlim(.45, 1)
ax.set_xlabel('tek degiskenli AUC (verim agirlikli, egitim kumesi)')
ax.set_title('kirmizi = literatur 8')
plt.tight_layout()
plt.show()

**GÖZLE:**

- En üstteki değişkenler hangileri? Fizik açısından beklediğin şey mi?
- `*_phi`, `*_eta` gibi değişkenlerin AUC'si $\approx0.5$ ise bilgi taşımıyorlar.
- Dikkat: **tek başına zayıf** bir değişken başka bir değişkenle birlikte güçlü
  olabilir (ör. `m_ll` bir kütle penceresi). Tek değişken AUC'si alt sınırdır,
  son karar değildir — Adım 10'daki permütasyon önemi bunu tamamlar.
- SR kesimi bir değişkeni zaten daraltmışsa (`dphi_Z_met > 2.5`), kesim sonrası
  AUC'si beklenenden düşük çıkabilir.

## Adım 8 — Korelasyon: aynı bilgiyi iki kez mi veriyoruz?

Birbirine çok bağlı ($|\rho|>0.9$) iki değişken modele çoğunlukla yeni bilgi
katmaz. Aşağıda **arka plan eğitim kümesinde** Pearson korelasyonu çizilir ve
güçlü çiftler listelenir; her çiftte tek değişkenli AUC'si düşük olan "atılabilir"
olarak önerilir.

**KARAR (isteğe bağlı):** `KORELASYON_LISTESI` ile hangi değişkenlerin ısı
haritasında görüneceğini seç (hepsi çok kalabalık olabilir).

In [ ]:
KORELASYON_LISTESI = tek.index[:16].tolist()   # en ayirici 16
ESIK_RHO = 0.9

C = b_tr[KORELASYON_LISTESI].corr()
n = len(KORELASYON_LISTESI)
fig, ax = plt.subplots(figsize=(0.55 * n + 2, 0.5 * n + 1))
im = ax.imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(n), KORELASYON_LISTESI, rotation=60, ha='right')
ax.set_yticks(range(n), KORELASYON_LISTESI)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{C.iloc[i, j]:+.1f}', ha='center', va='center',
                fontsize=7, color='white' if abs(C.iloc[i, j]) > .6 else 'k')
fig.colorbar(im, ax=ax, fraction=.046, label=r'Pearson $\rho$')
ax.set_title('arka plan korelasyonu (egitim)')
plt.tight_layout()
plt.show()

Ctum = b_tr[ADAY].corr()
cift = []
for i, a in enumerate(ADAY):
    for b in ADAY[i + 1:]:
        r = Ctum.loc[a, b]
        if abs(r) > ESIK_RHO:
            zayif = a if tek.AUC_verim[a] < tek.AUC_verim[b] else b
            cift.append({'a': a, 'b': b, 'rho': r, 'atilabilir': zayif})
print(f'|rho| > {ESIK_RHO} ciftler:')
print(pd.DataFrame(cift).round(3).to_string(index=False) if cift else 'yok')

**GÖZLE:** Pearson yalnızca **doğrusal** ilişkiyi ölçer. Örneğin `met` ve
`met_over_ptZ` doğrusal olarak orta düzeyde bağlı görünebilir ama oran, sahte
$E_T^{\rm miss}$'i bastıran farklı bir bilgi taşır. Listedeki öneri bir ipucudur,
fizik gerekçesi ağır basar.

## Adım 9 — KARAR: öznitelik seçimi

Adım 3, 7 ve 8'e bakarak kendi listeni yaz. Başlangıç olarak ana defterin
literatür 8'lisi konuldu. Örnek düşünceler:

- Tek değişkenli AUC'si $\approx0.5$ olanları çıkar.
- $|\rho|>0.9$ çiftlerinden birini bırak.
- Fizik gerekçesi olan bir değişken ekle (ör. `u_parallel`, `lep1_pt`).

Hücre seçimini kontrol eder ve uyarı verir; karar yine senindir.

In [ ]:
BENIM_OZNITELIKLERIM = [
    'met', 'mT', 'Z_pt', 'dphi_Z_met',
    'dR_ll', 'm_ll', 'n_jets', 'met_over_ptZ',
]

F = list(dict.fromkeys(BENIM_OZNITELIKLERIM))     # tekrarlari at
yok = [f for f in F if f not in bkg.columns]
assert not yok, f'boyle sutun yok: {yok}'
sabit_f = [f for f in F if b_tr[f].std() <= 1e-9]
assert not sabit_f, f'SR icinde sabit: {sabit_f}'

print(f'{len(F)} oznitelik secildi\n')
print(tek.loc[F, ['AUC_verim', 'AUC_ham', 'yon']].round(3).to_string())
Cs = b_tr[F].corr()
uyari = [(a, b, Cs.loc[a, b]) for i, a in enumerate(F) for b in F[i + 1:]
         if abs(Cs.loc[a, b]) > ESIK_RHO]
for a, b, r in uyari:
    print(f'uyari: {a} ile {b} cok bagli (rho={r:+.2f})')
zayif = tek.loc[F].query('AUC_verim < 0.55').index.tolist()
if zayif:
    print('uyari: tek basina zayif (AUC<0.55):', zayif,
          '-> Adim 10 onem grafigine bak')

## Adım 10 — Seçimini dene (yalnızca doğrulama kümesi)

Hızlı olduğu için yalnızca **BDT** kullanılır (ana defterle aynı ayar ve aynı
doğrulama-kaybıyla ağaç sayısı seçimi). Senin seçimin literatür 8'lisiyle
**doğrulama kümesinde** karşılaştırılır:

- `AUC_verim`, `AUC_ham`,
- `Z_A`: doğrulamada en iyi eşikteki Asimov anlamlılığı ($N_{\rm eff}\ge10$).

Ardından permütasyon önemi: her değişken karıştırılınca AUC ne kadar düşüyor?
**Düşüş $\approx0$ olan değişken** modele katkı vermiyor demektir → Adım 9'a dön,
çıkar, tekrar çalıştır.

In [ ]:
from sklearn.preprocessing import StandardScaler

NEFF_MIN = 10.0


def egit_bdt(F, seed=TOHUM, step=25, max_iter=600, patience=4):
    sc = StandardScaler().fit(b_tr[F])
    tf = lambda d: sc.transform(d[F])

    def xyw(bd, sd):
        x = np.vstack([tf(bd), tf(sd)])
        y = np.r_[np.zeros(len(bd)), np.ones(len(sd))]
        w = np.r_[np.full(len(bd), .5 / len(bd)), np.full(len(sd), .5 / len(sd))]
        return x, y, w

    x_tr, y_t, w_t = xyw(b_tr, s_tr)
    x_va, y_v, w_v = xyw(b_va, s_va)
    kur = lambda n, ws: HistGradientBoostingClassifier(
        max_depth=4, learning_rate=0.08, early_stopping=False,
        warm_start=ws, max_iter=n, random_state=seed)
    probe, best, best_n, waited = kur(step, True), np.inf, step, 0
    for n in range(step, max_iter + 1, step):
        probe.max_iter = n
        probe.fit(x_tr, y_t, sample_weight=w_t)
        loss = log_loss(y_v, probe.predict_proba(x_va)[:, 1], sample_weight=w_v)
        if loss < best - 1e-5:
            best, best_n, waited = loss, n, 0
        else:
            waited += 1
            if waited >= patience:
                break
    model = kur(best_n, False).fit(x_tr, y_t, sample_weight=w_t)
    skor = lambda d: 2 * model.predict_proba(tf(d))[:, 1] - 1
    return skor, best_n


def en_iyi_kesim(sb, ss, wb, ws, scale=1 / FRAC):
    cut, best = np.nan, -1.0
    for t in np.quantile(sb, np.linspace(0.5, 0.9995, 80)):
        w_sel = wb[sb > t]
        b, s = w_sel.sum() * LUMI * scale, ws[ss > t].sum() * LUMI * scale
        if s <= 0 or b <= 0 or n_eff(w_sel) < NEFF_MIN:
            continue
        z = asimov(s, b)
        if z > best:
            cut, best = t, z
    return cut, (best if best >= 0 else np.nan)


def degerlendir(skor, b, s):
    sb, ss = skor(b), skor(s)
    y = np.r_[np.zeros(len(sb)), np.ones(len(ss))]
    wb, ws = b.weight.to_numpy(), s.weight.to_numpy()
    cut, z = en_iyi_kesim(sb, ss, wb, ws)
    return {'AUC_verim': roc_auc_score(y, np.r_[sb, ss],
                                       sample_weight=np.abs(np.r_[wb, ws])),
            'AUC_ham': roc_auc_score(y, np.r_[sb, ss]), 'Z_A': z, 'esik': cut}


kiyas = {}
for ad, FF in [('literatur 8', LITERATUR_8), (f'benim {len(F)}', F)]:
    skor, n_agac = egit_bdt(FF)
    kiyas[ad] = {'d': len(FF), 'agac': n_agac, **degerlendir(skor, b_va, s_va)}
benim_skor = skor
print('DOGRULAMA kumesi:')
print(pd.DataFrame(kiyas).T.to_string())

In [ ]:
def perm_onem(skor, F, b, s, n_tekrar=5, seed=0):
    rng = np.random.default_rng(seed)
    y = np.r_[np.zeros(len(b)), np.ones(len(s))]
    w = np.abs(np.r_[b.weight, s.weight])
    taban = roc_auc_score(y, np.r_[skor(b), skor(s)], sample_weight=w)
    sonuc = {}
    for f in F:
        d = []
        for _ in range(n_tekrar):
            bc, sc_ = b.copy(), s.copy()
            tum = np.r_[bc[f].to_numpy(), sc_[f].to_numpy()]
            rng.shuffle(tum)
            bc[f], sc_[f] = tum[:len(bc)], tum[len(bc):]
            d.append(taban - roc_auc_score(y, np.r_[skor(bc), skor(sc_)],
                                           sample_weight=w))
        sonuc[f] = (np.mean(d), np.std(d))
    return sonuc


imp = perm_onem(benim_skor, F, b_va, s_va)
sira = sorted(F, key=lambda f: -imp[f][0])
fig, ax = plt.subplots(figsize=(0.7 * len(F) + 2, 3.8))
ax.bar(sira, [imp[f][0] for f in sira], yerr=[imp[f][1] for f in sira],
       color='#4878a8', capsize=3)
ax.axhline(0, color='k', lw=.8)
ax.set_xticks(range(len(sira)), sira, rotation=35, ha='right')
ax.set_ylabel('AUC dususu (verim agirlikli)')
ax.set_title('permutasyon onemi — senin secimin, dogrulama kumesi')
ax.grid(alpha=.3, axis='y')
plt.tight_layout()
plt.show()
for f in sira:
    print(f'  {f:22s} {imp[f][0]:+.4f} +- {imp[f][1]:.4f}')

**GÖZLE / KARAR:**

- Senin seçimin literatür 8'lisinden iyi mi kötü mü? Fark beş tohumdaki yayılımdan
  (ana defterde $Z_A$ için $\sim0.1$–$0.4$) büyük mü? Değilse "fark yok" demektir.
  Farklı `TOHUM` ile Adım 6'dan itibaren tekrar çalıştırarak yayılımı görebilirsin.
- Önemi $\approx0$ (hata çubuğu sıfırı kesen) değişkeni çıkarıp Adım 9'a dön.
- İki çok bağlı değişkenden biri çıkınca diğerinin önemi artar; bu normaldir.

Memnun kalınca Adım 11'e geç. **Adım 11'den sonra geri dönme.**

## Adım 11 — Son rapor: test kümesi (bir kez)

Eşik **doğrulama** kümesinde seçilir, test kümesinde sabit uygulanır (ana defter §4).

In [ ]:
def test_raporu(FF):
    skor, n_agac = egit_bdt(FF)
    v = degerlendir(skor, b_va, s_va)               # esik dogrulamada
    sb, ss = skor(b_te), skor(s_te)
    wb, ws = b_te.weight.to_numpy(), s_te.weight.to_numpy()
    y = np.r_[np.zeros(len(sb)), np.ones(len(ss))]
    kb, ks = sb > v['esik'], ss > v['esik']
    B, S = wb[kb].sum() * LUMI / FRAC, ws[ks].sum() * LUMI / FRAC
    ne = n_eff(wb[kb])
    return {'d': len(FF), 'AUC_verim': roc_auc_score(
                y, np.r_[sb, ss], sample_weight=np.abs(np.r_[wb, ws])),
            'AUC_ham': roc_auc_score(y, np.r_[sb, ss]),
            'esik': v['esik'], 'S': S, 'B': B, 'N_eff': ne,
            'Z_A': asimov(S, B) if ne >= NEFF_MIN else np.nan}


son = pd.DataFrame({'literatur 8': test_raporu(LITERATUR_8),
                    f'benim {len(F)}': test_raporu(F)}).T
print('TEST kumesi (BDT, tohum', TOHUM, '):')
print(son.round(3).to_string())
print('\nsecimin:', F)

**Sonraki adım.** Seçtiğin listeyi ana defterde (`monoz_dark_matter_analizi.ipynb`,
hücre 3) `FEATURES = [...]` satırına yaz. Böylece üç modeli (BDT, DNN, NSF) beş
tohumla, aynı protokolle çalıştırmış olursun. NSF'nin boyuta duyarlı olduğunu
unutma (ana defter §6a: 4D'de NSF $Z_A\approx3.8$, 8D'de $\approx1.8$).